# VIVO Backgammon — ENTRENAMIENTO GRANDE (self-play rápido) v2

**Pasos (Thomas):**
1. Importa este notebook desde el repo `Pirzl/Backgammon-AR-Pro`, rama `260816-gpu-train`.
2. Engranaje ⚙️ → Accelerator = **GPU P100**, Internet = **ON**. Save.
3. Run All.
4. Descarga `public/model_weights.json` y súbelo a InfinityFree en `dist/`.

El modelo de salida es compatible con el navegador tal cual.

In [ ]:
# Celda 0 — descarga TODO (forzado, sin cache)
import urllib.request, os
base = 'https://raw.githubusercontent.com/Pirzl/Backgammon-AR-Pro/260816-gpu-train/colab/'
for f in ['bg_engine.py', 'bg_fast.py', 'bg_train_fast.py', 'parity_test.py']:
    urllib.request.urlretrieve(base + f, f)
    print('descargado', f)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPU disponible:', gpus)
if not gpus:
    print('AVISO: no hay GPU -> lento. Cambia a GPU P100.')
else:
    print('OK: GPU lista.')

In [ ]:
# Celda 1 — test de paridad (integrado)
import subprocess
r = subprocess.run(['python', 'parity_test.py'], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print(r.stderr[-500:])
assert 'PARITY OK' in r.stdout, 'PARIDAD FALLÓ — no entrenes.'

In [ ]:
# Celda 2 — ENTRENAMIENTO GRANDE (self-play)
games = 100000
exploration = 0.15
max_moves = 400
epochs = 3
save_every = 10000
seed = 1
weights_arg = ''   # para reanudar: '--weights /kaggle/input/TU_INPUT/model_weights.json'

cmd = (
    f'python bg_train_fast.py --games {games} '
    f'--exploration {exploration} --max-moves {max_moves} '
    f'--epochs {epochs} --save-every {save_every} '
    f'--out public/model_weights.json --seed {seed} {weights_arg}'
)
print('Ejecutando:', cmd)
import subprocess
r = subprocess.run(cmd, shell=True)
print('exit code:', r.returncode)

In [ ]:
# Celda 3 — verifica y descarga
import os, json
p = 'public/model_weights.json'
assert os.path.exists(p), 'No se generó model_weights.json'
w = json.load(open(p))
print('Capas:', len(w))
print('Formas:', [l['shape'] for l in w])
print('Tamaño:', round(os.path.getsize(p)/1024/1024, 2), 'MB')
print('LISTO: descarga public/model_weights.json y súbelo a dist/ en InfinityFree')